# MDD Challenge 2025 - Data Exploration

This notebook explores all available data for Mispronunciation Detection and Diagnosis:

- `train.csv`: readable canonical/transcript text.
- `train_phones.csv`: canonical/transcript phoneme sequences, used directly for CTC training and evaluation.
- `lexicon_vmd.txt`: word-to-phoneme lexicon for coverage and consistency checks.
- `audio_data/train/*.wav`: 16 kHz mono audio.

EDA goal before training the baseline:

```text
audio -> phoneme CTC -> predicted transcript phones
```

## 1. Setup

In [ ]:
from pathlib import Path
import os
import re
import csv
import wave
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', 180)
pd.set_option('display.max_rows', 100)
plt.rcParams['figure.figsize'] = (10, 4)

RANDOM_STATE = 42

In [ ]:
def find_dataset_root():
    candidates = [
         Path('/kaggle/input/datasets/cquangnguynl/mdd-challenge/MDD-Challenge-2025-training-set'),
    ]
    for p in candidates:
        if (p / 'metadata' / 'train.csv').exists() and (p / 'metadata' / 'train_phones.csv').exists():
            return p

    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        for train_csv in kaggle_input.rglob('metadata/train.csv'):
            root = train_csv.parent.parent
            if (root / 'metadata' / 'train_phones.csv').exists():
                return root

    raise FileNotFoundError('Dataset root not found. Please set DATA_ROOT manually.')

DATA_ROOT = find_dataset_root()
META_DIR = DATA_ROOT / 'metadata'
AUDIO_DIR = DATA_ROOT / 'audio_data' / 'train'

TRAIN_CSV = META_DIR / 'train.csv'
TRAIN_PHONES_CSV = META_DIR / 'train_phones.csv'
LEXICON_PATH = META_DIR / 'lexicon_vmd.txt'

print('DATA_ROOT:', DATA_ROOT)
print('TRAIN_CSV:', TRAIN_CSV)
print('TRAIN_PHONES_CSV:', TRAIN_PHONES_CSV)
print('LEXICON_PATH:', LEXICON_PATH, 'exists=', LEXICON_PATH.exists())
print('AUDIO_DIR:', AUDIO_DIR, 'exists=', AUDIO_DIR.exists())

## 2. Load Metadata

In [ ]:
train_text = pd.read_csv(TRAIN_CSV)
train_phones = pd.read_csv(TRAIN_PHONES_CSV)

required_cols = {'id', 'path', 'canonical', 'transcript'}
assert required_cols.issubset(train_text.columns), train_text.columns
assert required_cols.issubset(train_phones.columns), train_phones.columns

print('train_text shape:', train_text.shape)
print('train_phones shape:', train_phones.shape)
print('train_text columns:', list(train_text.columns))
print('train_phones columns:', list(train_phones.columns))

display(train_text.head())
display(train_phones.head())

In [ ]:
meta_checks = pd.DataFrame({
    'source': ['train_text', 'train_phones'],
    'rows': [len(train_text), len(train_phones)],
    'missing_id': [train_text['id'].isna().sum(), train_phones['id'].isna().sum()],
    'missing_path': [train_text['path'].isna().sum(), train_phones['path'].isna().sum()],
    'missing_canonical': [train_text['canonical'].isna().sum(), train_phones['canonical'].isna().sum()],
    'missing_transcript': [train_text['transcript'].isna().sum(), train_phones['transcript'].isna().sum()],
    'duplicated_id': [train_text['id'].duplicated().sum(), train_phones['id'].duplicated().sum()],
})
display(meta_checks)

print('Same row order by id:', train_text['id'].equals(train_phones['id']))
print('Same row order by path:', train_text['path'].equals(train_phones['path']))

merged = train_text[['id', 'path', 'canonical', 'transcript']].merge(
    train_phones[['id', 'path', 'canonical', 'transcript']],
    on=['id', 'path'],
    how='outer',
    suffixes=('_text', '_phones'),
    indicator=True,
)
print('Merge indicator:')
display(merged['_merge'].value_counts(dropna=False))
display(merged.head())

## 3. Normalization Helpers

`evaluate.py` computes metrics by removing `*`, removing `$`, then calling `.split()`. Use the same logic for EDA and label preparation.

In [ ]:
def normalize_phone_text(s):
    return ' '.join(str(s).replace('*', '').replace('$', '').split())

def eval_phone_tokens(s):
    return normalize_phone_text(s).split()

def word_tokens(s):
    return str(s).strip().split()

for df in [train_phones]:
    df['canonical_norm'] = df['canonical'].map(normalize_phone_text)
    df['transcript_norm'] = df['transcript'].map(normalize_phone_text)
    df['canonical_has_dollar'] = df['canonical'].astype(str).str.contains(r'\$', regex=True)
    df['transcript_has_dollar'] = df['transcript'].astype(str).str.contains(r'\$', regex=True)

print('Rows with $ in train_phones/canonical:', int(train_phones['canonical_has_dollar'].sum()))
print('Rows with $ in train_phones/transcript:', int(train_phones['transcript_has_dollar'].sum()))
print('Rows changed by phone normalization/canonical:', int((train_phones['canonical'] != train_phones['canonical_norm']).sum()))
print('Rows changed by phone normalization/transcript:', int((train_phones['transcript'] != train_phones['transcript_norm']).sum()))

## 4. Audio Path And File Checks

In [ ]:
def resolve_audio_path(rel_path):
    rel = Path(str(rel_path))
    candidates = [
        DATA_ROOT / rel,
        DATA_ROOT / 'audio_data' / 'train' / rel.name,
        AUDIO_DIR / rel.name,
    ]
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]

train_text['audio_path'] = train_text['path'].map(resolve_audio_path)
train_phones['audio_path'] = train_phones['path'].map(resolve_audio_path)
train_text['audio_exists'] = train_text['audio_path'].map(lambda p: p.exists())
train_phones['audio_exists'] = train_phones['audio_path'].map(lambda p: p.exists())

print('Audio exists counts:')
display(train_text['audio_exists'].value_counts(dropna=False))
display(train_text.loc[~train_text['audio_exists'], ['id', 'path', 'audio_path']].head())

n_wav_files = len(list(AUDIO_DIR.glob('*.wav'))) if AUDIO_DIR.exists() else 0
print('WAV files in audio train dir:', n_wav_files)
print('Metadata rows:', len(train_text))

## 5. Adult vs Child Rule

Current rule: file/id with a long timestamp suffix is `adult`; otherwise it is `child`.

In [ ]:
def infer_speaker_group(sample_id, rel_path):
    stem = Path(str(rel_path)).stem
    has_long_timestamp_suffix = bool(re.search(r'[_-]\d{10,}$', stem)) or bool(re.search(r'[_-]\d{10,}$', str(sample_id)))
    return 'adult' if has_long_timestamp_suffix else 'child'

train_text['speaker_group'] = [infer_speaker_group(i, p) for i, p in zip(train_text['id'], train_text['path'])]
train_phones['speaker_group'] = train_text['speaker_group']

print('Adult/child counts:')
display(train_text['speaker_group'].value_counts().rename_axis('speaker_group').reset_index(name='samples'))
print('Adult/child ratio:')
display(train_text['speaker_group'].value_counts(normalize=True).rename_axis('speaker_group').reset_index(name='ratio'))

display(train_text[['id', 'path', 'speaker_group', 'canonical', 'transcript']].sample(20, random_state=RANDOM_STATE))

## 6. Readable Text EDA

In [ ]:
for col in ['canonical', 'transcript']:
    train_text[f'{col}_n_words'] = train_text[col].map(lambda s: len(word_tokens(s)))

train_text['text_equal'] = train_text['canonical'] == train_text['transcript']
train_text['word_len_delta'] = train_text['transcript_n_words'] - train_text['canonical_n_words']

summary_cols = ['canonical_n_words', 'transcript_n_words', 'word_len_delta']
display(train_text[summary_cols].describe())
print('Readable canonical == transcript ratio:', train_text['text_equal'].mean())
print('Rows with readable mismatch:', int((~train_text['text_equal']).sum()))

display(train_text.loc[~train_text['text_equal'], ['id', 'speaker_group', 'canonical', 'transcript']].head(30))

In [ ]:
word_vocab = Counter(tok for s in pd.concat([train_text['canonical'], train_text['transcript']]) for tok in word_tokens(s))
print('Readable word vocab size:', len(word_vocab))
display(pd.DataFrame(word_vocab.most_common(50), columns=['word', 'count']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_text['canonical_n_words'].hist(bins=30, ax=axes[0])
axes[0].set_title('Canonical word length')
axes[0].set_xlabel('words')
axes[0].set_ylabel('samples')
train_text['word_len_delta'].hist(bins=25, ax=axes[1])
axes[1].set_title('Transcript words - canonical words')
axes[1].set_xlabel('delta')
plt.tight_layout()
plt.show()

## 7. Phoneme EDA

In [ ]:
for col in ['canonical_norm', 'transcript_norm']:
    train_phones[f'{col}_n_phones'] = train_phones[col].map(lambda s: len(eval_phone_tokens(s)))

train_phones['phone_equal'] = train_phones['canonical_norm'] == train_phones['transcript_norm']
train_phones['phone_len_delta'] = train_phones['transcript_norm_n_phones'] - train_phones['canonical_norm_n_phones']

phone_summary_cols = ['canonical_norm_n_phones', 'transcript_norm_n_phones', 'phone_len_delta']
display(train_phones[phone_summary_cols].describe())
print('Phone canonical == transcript ratio:', train_phones['phone_equal'].mean())
print('Rows with phone mismatch:', int((~train_phones['phone_equal']).sum()))

display(train_phones.loc[~train_phones['phone_equal'], ['id', 'speaker_group', 'canonical_norm', 'transcript_norm']].head(30))

In [ ]:
phone_vocab = Counter(tok for s in pd.concat([train_phones['canonical_norm'], train_phones['transcript_norm']]) for tok in eval_phone_tokens(s))
transcript_phone_vocab = Counter(tok for s in train_phones['transcript_norm'] for tok in eval_phone_tokens(s))
canonical_phone_vocab = Counter(tok for s in train_phones['canonical_norm'] for tok in eval_phone_tokens(s))

print('Phone vocab size total:', len(phone_vocab))
print('Phone vocab size canonical:', len(canonical_phone_vocab))
print('Phone vocab size transcript:', len(transcript_phone_vocab))
print('Phones only in transcript:', sorted(set(transcript_phone_vocab) - set(canonical_phone_vocab)))
print('Phones only in canonical:', sorted(set(canonical_phone_vocab) - set(transcript_phone_vocab)))

display(pd.DataFrame(phone_vocab.most_common(80), columns=['phone', 'count']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_phones['canonical_norm_n_phones'].hist(bins=40, ax=axes[0])
axes[0].set_title('Canonical phone length')
axes[0].set_xlabel('phones')
axes[0].set_ylabel('samples')
train_phones['phone_len_delta'].hist(bins=31, ax=axes[1])
axes[1].set_title('Transcript phones - canonical phones')
axes[1].set_xlabel('delta')
plt.tight_layout()
plt.show()

## 8. Evaluator-Style Alignment Statistics

This section mirrors `_align_pair()` in `evaluate.py` to summarize true errors between `canonical` and `transcript` at phoneme level.

In [ ]:
def align(seq1, seq2):
    GAP = -1
    MATCH = 1
    MISMATCH = -1
    n, m = len(seq1), len(seq2)
    score = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        score[i][0] = GAP * i
    for j in range(n + 1):
        score[0][j] = GAP * j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if seq1[j - 1] == seq2[i - 1]:
                s = MATCH
            elif seq1[j - 1] == '<eps>' or seq2[i - 1] == '<eps>':
                s = GAP
            else:
                s = MISMATCH
            score[i][j] = max(score[i - 1][j - 1] + s, score[i - 1][j] + GAP, score[i][j - 1] + GAP)

    align1, align2 = [], []
    i, j = m, n
    while i > 0 and j > 0:
        if seq1[j - 1] == seq2[i - 1]:
            s = MATCH
        elif seq1[j - 1] == '<eps>' or seq2[i - 1] == '<eps>':
            s = GAP
        else:
            s = MISMATCH
        if score[i][j] == score[i - 1][j - 1] + s:
            align1.append(seq1[j - 1]); align2.append(seq2[i - 1])
            i -= 1; j -= 1
        elif score[i][j] == score[i][j - 1] + GAP:
            align1.append(seq1[j - 1]); align2.append('<eps>')
            j -= 1
        else:
            align1.append('<eps>'); align2.append(seq2[i - 1])
            i -= 1
    while j > 0:
        align1.append(seq1[j - 1]); align2.append('<eps>'); j -= 1
    while i > 0:
        align1.append('<eps>'); align2.append(seq2[i - 1]); i -= 1
    align1.reverse(); align2.reverse()
    return align1, align2

def ops(aligned1, aligned2):
    out = []
    for r, h in zip(aligned1, aligned2):
        if r != '<eps>' and h == '<eps>':
            out.append('D')
        elif r == '<eps>' and h != '<eps>':
            out.append('I')
        elif r != h:
            out.append('S')
        else:
            out.append('C')
    return out

def align_pair(s1, s2):
    a1, a2 = align(eval_phone_tokens(s1), eval_phone_tokens(s2))
    return a1, a2, ops(a1, a2)

def row_alignment_stats(row):
    a_ref, a_hyp, op = align_pair(row['canonical_norm'], row['transcript_norm'])
    c = Counter(op)
    return pd.Series({
        'n_correct': c['C'],
        'n_sub': c['S'],
        'n_del': c['D'],
        'n_ins': c['I'],
        'n_errors': c['S'] + c['D'] + c['I'],
        'ref_len_eval': c['C'] + c['S'] + c['D'],
    })

align_stats = train_phones.apply(row_alignment_stats, axis=1)
train_phones = pd.concat([train_phones, align_stats], axis=1)
train_phones['has_pron_error'] = train_phones['n_errors'] > 0
train_phones['true_per_like'] = train_phones['n_errors'] / train_phones['ref_len_eval'].replace(0, np.nan)

display(train_phones[['n_correct', 'n_sub', 'n_del', 'n_ins', 'n_errors', 'ref_len_eval', 'true_per_like']].describe())
print('Rows with at least one pronunciation error:', train_phones['has_pron_error'].mean())
print('Total alignment ops:', train_phones[['n_sub', 'n_del', 'n_ins', 'n_correct']].sum().to_dict())

In [ ]:
group_summary = train_phones.groupby('speaker_group').agg(
    samples=('id', 'count'),
    error_rate=('has_pron_error', 'mean'),
    avg_errors=('n_errors', 'mean'),
    avg_true_per_like=('true_per_like', 'mean'),
    avg_canonical_phones=('canonical_norm_n_phones', 'mean'),
)
display(group_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_text['speaker_group'].value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Adult vs child sample count')
axes[0].set_xlabel('speaker group')
axes[0].set_ylabel('samples')
axes[0].tick_params(axis='x', rotation=0)

train_phones[['n_sub', 'n_del', 'n_ins']].sum().plot(kind='bar', ax=axes[1])
axes[1].set_title('True pronunciation error operation counts')
axes[1].set_ylabel('count')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

train_phones['n_errors'].value_counts().sort_index().head(40).plot(kind='bar', figsize=(12, 4))
plt.title('Number of true phone errors per utterance')
plt.xlabel('errors')
plt.ylabel('samples')
plt.show()

In [ ]:
def collect_error_pairs(df):
    rows = []
    for _, row in df.iterrows():
        a_ref, a_hyp, op = align_pair(row['canonical_norm'], row['transcript_norm'])
        for ref, hyp, o in zip(a_ref, a_hyp, op):
            if o != 'C':
                rows.append({
                    'id': row['id'],
                    'speaker_group': row['speaker_group'],
                    'op': o,
                    'canonical_phone': ref,
                    'transcript_phone': hyp,
                })
    return pd.DataFrame(rows)

error_pairs = collect_error_pairs(train_phones)
print('Total aligned error tokens:', len(error_pairs))
display(error_pairs.head(20))

print('Most common substitution pairs:')
display(error_pairs.query("op == 'S'").value_counts(['canonical_phone', 'transcript_phone']).reset_index(name='count').head(30))

print('Most common deletions:')
display(error_pairs.query("op == 'D'").value_counts(['canonical_phone']).reset_index(name='count').head(30))

print('Most common insertions:')
display(error_pairs.query("op == 'I'").value_counts(['transcript_phone']).reset_index(name='count').head(30))

## 9. Lexicon EDA

The lexicon is not required for the CTC baseline because `train_phones.csv/transcript` is already the target. Still, it is useful for coverage and consistency checks between readable text and phone sequences.

In [ ]:
def load_lexicon(path):
    lex = {}
    duplicate_words = Counter()
    malformed = []
    with open(path, encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 2:
                malformed.append((line_no, line))
                continue
            word, phones = parts[0], parts[1:]
            if word in lex:
                duplicate_words[word] += 1
            lex[word] = phones
    return lex, duplicate_words, malformed

lexicon, duplicate_lex_words, malformed_lex_lines = load_lexicon(LEXICON_PATH)
print('Lexicon entries:', len(lexicon))
print('Duplicate lexicon words:', len(duplicate_lex_words))
print('Malformed lines:', len(malformed_lex_lines))
print('First malformed lines:', malformed_lex_lines[:5])

display(pd.DataFrame(list(lexicon.items())[:20], columns=['word', 'phones']))

In [ ]:
def lexicon_convert_text(text, lexicon):
    phones = []
    oov = []
    for w in word_tokens(text):
        if w in lexicon:
            phones.extend(lexicon[w])
        else:
            oov.append(w)
    return ' '.join(phones), oov

all_text_words = Counter(tok for s in pd.concat([train_text['canonical'], train_text['transcript']]) for tok in word_tokens(s))
oov_words = {w: c for w, c in all_text_words.items() if w not in lexicon}
print('Readable text vocab size:', len(all_text_words))
print('OOV words vs lexicon:', len(oov_words))
display(pd.DataFrame(Counter(oov_words).most_common(50), columns=['word', 'count']))

coverage_rows = []
for source_col in ['canonical', 'transcript']:
    n_rows_with_oov = 0
    n_oov_tokens = 0
    for text in train_text[source_col]:
        _, oov = lexicon_convert_text(text, lexicon)
        if oov:
            n_rows_with_oov += 1
            n_oov_tokens += len(oov)
    coverage_rows.append({'text_col': source_col, 'rows_with_oov': n_rows_with_oov, 'oov_tokens': n_oov_tokens})
display(pd.DataFrame(coverage_rows))

In [ ]:
# Compare lexicon-generated phone strings with train_phones.
# Strict comparison after evaluate-style normalization.
compare_rows = []
for idx in range(len(train_text)):
    row_text = train_text.iloc[idx]
    row_phone = train_phones.iloc[idx]
    for col in ['canonical', 'transcript']:
        generated, oov = lexicon_convert_text(row_text[col], lexicon)
        generated_norm = normalize_phone_text(generated)
        target_norm = row_phone[f'{col}_norm']
        compare_rows.append({
            'id': row_text['id'],
            'col': col,
            'has_oov': bool(oov),
            'oov': ' '.join(oov),
            'generated_norm': generated_norm,
            'target_norm': target_norm,
            'match': generated_norm == target_norm,
        })

lex_compare = pd.DataFrame(compare_rows)
print('Lexicon-generated phones exact match rate:')
display(lex_compare.groupby('col')['match'].mean())
print('Mismatch counts:')
display(lex_compare.groupby(['col', 'has_oov'])['match'].agg(['count', 'mean']))

display(lex_compare.loc[~lex_compare['match'], ['id', 'col', 'has_oov', 'oov', 'generated_norm', 'target_norm']].head(20))

## 10. Audio Duration And Format

In [ ]:
def wav_info(path):
    try:
        with wave.open(str(path), 'rb') as wf:
            n_frames = wf.getnframes()
            sr = wf.getframerate()
            channels = wf.getnchannels()
            sampwidth = wf.getsampwidth()
        return pd.Series({
            'sample_rate': sr,
            'channels': channels,
            'sample_width_bytes': sampwidth,
            'duration_sec': n_frames / sr if sr else np.nan,
        })
    except Exception:
        return pd.Series({
            'sample_rate': np.nan,
            'channels': np.nan,
            'sample_width_bytes': np.nan,
            'duration_sec': np.nan,
        })

audio_info = train_text['audio_path'].apply(wav_info)
train_text = pd.concat([train_text, audio_info], axis=1)
train_phones[['sample_rate', 'channels', 'sample_width_bytes', 'duration_sec']] = train_text[['sample_rate', 'channels', 'sample_width_bytes', 'duration_sec']]

display(train_text[['sample_rate', 'channels', 'sample_width_bytes', 'duration_sec']].describe())
print('Sample-rate counts:')
display(train_text['sample_rate'].value_counts(dropna=False))
print('Channel counts:')
display(train_text['channels'].value_counts(dropna=False))
print('Sample width counts:')
display(train_text['sample_width_bytes'].value_counts(dropna=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_text['duration_sec'].hist(bins=50, ax=axes[0])
axes[0].set_title('Audio duration')
axes[0].set_xlabel('seconds')
axes[0].set_ylabel('samples')

axes[1].scatter(train_phones['canonical_norm_n_phones'], train_text['duration_sec'], alpha=0.35, s=12)
axes[1].set_title('Duration vs canonical phone count')
axes[1].set_xlabel('phones')
axes[1].set_ylabel('seconds')
plt.tight_layout()
plt.show()

display(train_text.nlargest(10, 'duration_sec')[['id', 'speaker_group', 'duration_sec', 'canonical', 'transcript']])
display(train_text.nsmallest(10, 'duration_sec')[['id', 'speaker_group', 'duration_sec', 'canonical', 'transcript']])

## 11. Listen To Samples

In [ ]:
from IPython.display import Audio, display

def read_wav_pcm(path):
    with wave.open(str(path), 'rb') as wf:
        sr = wf.getframerate()
        n_channels = wf.getnchannels()
        sampwidth = wf.getsampwidth()
        frames = wf.readframes(wf.getnframes())
    if sampwidth == 2:
        data = np.frombuffer(frames, dtype=np.int16).astype(np.float32) / 32768.0
    elif sampwidth == 4:
        data = np.frombuffer(frames, dtype=np.int32).astype(np.float32) / 2147483648.0
    elif sampwidth == 1:
        data = (np.frombuffer(frames, dtype=np.uint8).astype(np.float32) - 128.0) / 128.0
    else:
        raise ValueError(f'Unsupported sample width: {sampwidth}')
    if n_channels > 1:
        data = data.reshape(-1, n_channels).mean(axis=1)
    return data, sr

def show_sample(row_idx=None, only_error=False, speaker_group=None):
    subset = train_phones
    if only_error:
        subset = subset[subset['has_pron_error']]
    if speaker_group is not None:
        subset = subset[subset['speaker_group'] == speaker_group]
    if row_idx is None:
        row = subset.sample(1, random_state=None).iloc[0]
    else:
        row = train_phones.iloc[row_idx]

    text_row = train_text.loc[train_text['id'] == row['id']].iloc[0]
    path = resolve_audio_path(row['path'])
    print('id:', row['id'])
    print('speaker_group:', row['speaker_group'])
    print('path:', path)
    print('duration:', row.get('duration_sec', None))
    print('canonical text:', text_row['canonical'])
    print('transcript text:', text_row['transcript'])
    print('canonical phones:', row['canonical_norm'])
    print('transcript phones:', row['transcript_norm'])
    print('alignment ops:', {k: int(row[k]) for k in ['n_sub', 'n_del', 'n_ins', 'n_errors']})

    y, sr = read_wav_pcm(path)
    t = np.arange(len(y)) / sr
    plt.figure(figsize=(12, 3))
    plt.plot(t, y, linewidth=0.7)
    plt.title(str(row['id']))
    plt.xlabel('seconds')
    plt.ylabel('amplitude')
    plt.tight_layout()
    plt.show()
    display(Audio(y, rate=sr))

show_sample(only_error=True)

## 12. Training-Oriented Artifacts

The cells below create the phoneme vocabulary and baseline CSV files for pipeline/evaluator sanity checks.

In [ ]:
# CTC vocabulary: each phoneme token is one class. Do not split IPA/tone tokens into characters.
phone_tokens_sorted = sorted(transcript_phone_vocab.keys())
blank_token = '<blank>'
unk_token = '<unk>'

id2phone = [blank_token, unk_token] + phone_tokens_sorted
phone2id = {p: i for i, p in enumerate(id2phone)}

print('CTC vocab size including blank/unk:', len(id2phone))
print('blank id:', phone2id[blank_token])
print('unk id:', phone2id[unk_token])
display(pd.DataFrame({'id': range(len(id2phone)), 'phone': id2phone}).head(80))

# Example encode/decode target labels.
def encode_phone_sequence(s):
    return [phone2id.get(tok, phone2id[unk_token]) for tok in eval_phone_tokens(s)]

def decode_phone_ids(ids, collapse_ctc=False):
    out = []
    prev = None
    for idx in ids:
        if collapse_ctc and idx == prev:
            prev = idx
            continue
        prev = idx
        if idx == phone2id[blank_token]:
            continue
        out.append(id2phone[idx] if idx < len(id2phone) else unk_token)
    return ' '.join(out)

example = train_phones.iloc[0]['transcript_norm']
encoded = encode_phone_sequence(example)
print('Example transcript:', example)
print('Encoded:', encoded[:30], '... len=', len(encoded))
print('Decoded:', decode_phone_ids(encoded))

In [ ]:
OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

ground_truth_path = OUT_DIR / 'ground_truth_train_phones.csv'
baseline_canonical_path = OUT_DIR / 'baseline_predict_canonical.csv'
oracle_transcript_path = OUT_DIR / 'oracle_predict_transcript.csv'
phone_vocab_path = OUT_DIR / 'phone_vocab.csv'

train_phones[['canonical_norm', 'transcript_norm']].rename(
    columns={'canonical_norm': 'canonical', 'transcript_norm': 'transcript'}
).to_csv(ground_truth_path, index=False)

train_phones[['canonical_norm']].rename(columns={'canonical_norm': 'predict'}).to_csv(baseline_canonical_path, index=False)
train_phones[['transcript_norm']].rename(columns={'transcript_norm': 'predict'}).to_csv(oracle_transcript_path, index=False)
pd.DataFrame({'id': range(len(id2phone)), 'phone': id2phone}).to_csv(phone_vocab_path, index=False)

print('Wrote:')
print(ground_truth_path)
print(baseline_canonical_path)
print(oracle_transcript_path)
print(phone_vocab_path)

## 13. Training Notes

Recommended baseline:

```text
input  = audio waveform, 16 kHz mono
target = train_phones/transcript_norm
model  = Vietnamese wav2vec2 or HuBERT encoder + phoneme CTC head
output = results.csv with one column: predict
```

Practical notes:

- Do not add `$`; current data has no word boundary and the evaluator removes `$` anyway.
- Do not use `train.csv/transcript` as the main target for the phoneme CTC baseline; use `train_phones.csv/transcript`.
- Use `train.csv` for EDA, debugging, and readable error analysis.
- Use `lexicon_vmd.txt` for coverage/consistency checks or later advanced decoding.
- Audio is already 16 kHz mono, so no resampling is needed.
- Validation split should be careful and include child/adult where possible, but adult has very few samples, so use it mainly as a diagnostic slice.
- Augmentation should be light and audio-only: speed/gain/noise/SpecAugment; avoid cropping because it breaks label alignment.

## 14. Baseline Config

Sanity-first baseline: replace the original text CTC head with a new phoneme CTC head. Keep this run small first; after the pipeline is verified, set `USE_SUBSET = False` and increase epochs.

In [ ]:
import math
import random
import sys
import subprocess
import importlib.util
import inspect
from dataclasses import dataclass

def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        package = pip_name or import_name
        print(f'Installing missing package: {package}')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

ensure_package('torch')
ensure_package('sklearn', 'scikit-learn')
ensure_package('transformers')
ensure_package('accelerate')

import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoFeatureExtractor,
    Trainer,
    TrainingArguments,
    Wav2Vec2ForCTC,
    get_linear_schedule_with_warmup,
    set_seed,
)

BASELINE_SEED = 42
set_seed(BASELINE_SEED)
random.seed(BASELINE_SEED)
np.random.seed(BASELINE_SEED)

MODEL_NAME = 'nguyenvulebinh/wav2vec2-base-vietnamese-250h'
SAMPLING_RATE = 16000

# FAST_DEV_RUN is only for checking that the notebook runs end-to-end.
# The phoneme CTC head is randomly initialized; a 1-epoch subset run usually collapses to blank.
FAST_DEV_RUN = False
USE_SUBSET = FAST_DEV_RUN
MAX_TRAIN_SAMPLES = 800
MAX_VALID_SAMPLES = 200
NUM_EPOCHS = 2 if FAST_DEV_RUN else 20

PER_DEVICE_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 2
# The phoneme CTC head is randomly initialized, so it needs a larger LR than the pretrained encoder.
ENCODER_LEARNING_RATE = 2e-5
HEAD_LEARNING_RATE = 2e-3
LEARNING_RATE = ENCODER_LEARNING_RATE
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
FP16 = torch.cuda.is_available()
FREEZE_FEATURE_EXTRACTOR = True
DISABLE_MODEL_SPEC_AUGMENT = True

# Greedy CTC can be over-conservative with blank. This subtracts from the blank logit only at decode time.
# Set to 0.0 for raw greedy decoding; tune on validation if predictions are still empty.
BLANK_DECODE_PENALTY = 0.5

ENABLE_AUDIO_AUGMENT = False
AUG_GAIN_DB = 6.0
AUG_NOISE_PROB = 0.25
AUG_NOISE_SNR_DB = (15.0, 30.0)

RUN_DIR = OUT_DIR / 'phoneme_ctc_baseline'
RUN_DIR.mkdir(parents=True, exist_ok=True)

print('MODEL_NAME:', MODEL_NAME)
print('RUN_DIR:', RUN_DIR)
print('FAST_DEV_RUN:', FAST_DEV_RUN)
print('NUM_EPOCHS:', NUM_EPOCHS)
print('BLANK_DECODE_PENALTY:', BLANK_DECODE_PENALTY)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 15. Train/Validation Split

The split is stratified by `speaker_group` and true error presence. Adult has very few samples, so it is diagnostic only.

In [ ]:
baseline_df = train_phones.copy().reset_index(drop=True)
baseline_df['audio_path_str'] = baseline_df['audio_path'].map(str)
baseline_df['label_text'] = baseline_df['transcript_norm']
baseline_df['stratify_key'] = baseline_df['speaker_group'].astype(str) + '_' + baseline_df['has_pron_error'].astype(int).astype(str)

# Collapse very small strata if needed; sklearn stratify requires at least 2 examples per class.
stratum_counts = baseline_df['stratify_key'].value_counts()
rare_strata = set(stratum_counts[stratum_counts < 2].index)
baseline_df.loc[baseline_df['stratify_key'].isin(rare_strata), 'stratify_key'] = baseline_df.loc[
    baseline_df['stratify_key'].isin(rare_strata), 'speaker_group'
].astype(str)

train_idx, valid_idx = train_test_split(
    np.arange(len(baseline_df)),
    test_size=0.15,
    random_state=BASELINE_SEED,
    stratify=baseline_df['stratify_key'],
)

train_df = baseline_df.iloc[train_idx].reset_index(drop=True)
valid_df = baseline_df.iloc[valid_idx].reset_index(drop=True)

if USE_SUBSET:
    train_df = train_df.sample(min(MAX_TRAIN_SAMPLES, len(train_df)), random_state=BASELINE_SEED).reset_index(drop=True)
    valid_df = valid_df.sample(min(MAX_VALID_SAMPLES, len(valid_df)), random_state=BASELINE_SEED).reset_index(drop=True)

split_dir = RUN_DIR / 'splits'
split_dir.mkdir(parents=True, exist_ok=True)
train_df[['id', 'path', 'speaker_group', 'has_pron_error']].to_csv(split_dir / 'train_split_ids.csv', index=False)
valid_df[['id', 'path', 'speaker_group', 'has_pron_error']].to_csv(split_dir / 'valid_split_ids.csv', index=False)

print('train rows:', len(train_df))
print('valid rows:', len(valid_df))
print('split dir:', split_dir)
print('\ntrain speaker_group:')
display(train_df['speaker_group'].value_counts(dropna=False))
print('\nvalid speaker_group:')
display(valid_df['speaker_group'].value_counts(dropna=False))
print('\ntrain has_pron_error:')
display(train_df['has_pron_error'].value_counts(dropna=False))
print('\nvalid has_pron_error:')
display(valid_df['has_pron_error'].value_counts(dropna=False))

## 16. Dataset And Collator

No resampling is needed because all WAV files are 16 kHz mono. Labels are phoneme IDs; padding labels use `-100` for CTC loss masking.

In [ ]:
def apply_light_audio_augment(y):
    if not ENABLE_AUDIO_AUGMENT:
        return y
    y = y.copy()
    gain_db = np.random.uniform(-AUG_GAIN_DB, AUG_GAIN_DB)
    y *= float(10 ** (gain_db / 20.0))
    if np.random.rand() < AUG_NOISE_PROB:
        signal_power = float(np.mean(y ** 2) + 1e-10)
        snr_db = np.random.uniform(*AUG_NOISE_SNR_DB)
        noise_power = signal_power / float(10 ** (snr_db / 10.0))
        y += np.random.normal(0.0, math.sqrt(noise_power), size=y.shape).astype(np.float32)
    return np.clip(y, -1.0, 1.0).astype(np.float32)

class PhonemeCtcDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        y, sr = read_wav_pcm(row['audio_path'])
        if sr != SAMPLING_RATE:
            raise ValueError(f'Expected {SAMPLING_RATE} Hz, got {sr}: {row["audio_path"]}')
        y = y.astype(np.float32)
        if self.augment:
            y = apply_light_audio_augment(y)
        labels = encode_phone_sequence(row['label_text'])
        return {
            'input_values': y,
            'labels': labels,
            'id': row['id'],
        }

@dataclass
class CtcDataCollator:
    feature_extractor: object
    sampling_rate: int = SAMPLING_RATE
    label_pad_id: int = -100

    def __call__(self, features):
        input_values = [f['input_values'] for f in features]
        batch = self.feature_extractor(
            input_values,
            sampling_rate=self.sampling_rate,
            padding=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        labels = [torch.tensor(f['labels'], dtype=torch.long) for f in features]
        max_len = max(len(x) for x in labels)
        padded = torch.full((len(labels), max_len), self.label_pad_id, dtype=torch.long)
        for i, label in enumerate(labels):
            padded[i, : len(label)] = label
        batch['labels'] = padded
        return batch

feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
train_dataset = PhonemeCtcDataset(train_df, augment=ENABLE_AUDIO_AUGMENT)
valid_dataset = PhonemeCtcDataset(valid_df, augment=False)
data_collator = CtcDataCollator(feature_extractor=feature_extractor)

sample_item = train_dataset[0]
print('sample audio shape:', sample_item['input_values'].shape)
print('sample label len:', len(sample_item['labels']))
print('sample decoded label:', decode_phone_ids(sample_item['labels']))

## 17. Model Setup

The pretrained text CTC head is intentionally replaced. The encoder weights are reused; the new head predicts phoneme IDs.

In [ ]:
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    vocab_size=len(id2phone),
    pad_token_id=phone2id[blank_token],
    ctc_loss_reduction='mean',
    ctc_zero_infinity=True,
    ignore_mismatched_sizes=True,
)

model.config.pad_token_id = phone2id[blank_token]
model.config.ctc_zero_infinity = True
model.config.ctc_loss_reduction = 'mean'

if DISABLE_MODEL_SPEC_AUGMENT:
    model.config.apply_spec_augment = False
    model.config.mask_time_prob = 0.0

if FREEZE_FEATURE_EXTRACTOR:
    if hasattr(model, 'freeze_feature_encoder'):
        model.freeze_feature_encoder()
    else:
        model.freeze_feature_extractor()

# Quick CTC sanity check: target lengths must fit inside the model's frame lengths.
with torch.no_grad():
    sample_batch = data_collator([train_dataset[i] for i in range(min(2, len(train_dataset)))])
    sample_loss = model(**sample_batch).loss

print('vocab_size:', model.config.vocab_size)
print('blank/pad id:', model.config.pad_token_id)
print('lm_head:', model.lm_head)
print('lm_head out features:', model.lm_head.out_features)
print('phoneme vocab size:', len(id2phone))
print('feature encoder frozen:', FREEZE_FEATURE_EXTRACTOR)
print('model spec augment disabled:', DISABLE_MODEL_SPEC_AUGMENT)
print('initial sample CTC loss:', float(sample_loss.detach().cpu()))

## 18. Fine-Tuning

This is configured as a short sanity run. Increase samples/epochs after the first successful metric run.

In [ ]:
training_kwargs = dict(
    output_dir=str(RUN_DIR / 'checkpoints'),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    fp16=FP16,
    logging_steps=10,
    save_strategy='epoch',
    save_total_limit=1,
    report_to='none',
    remove_unused_columns=False,
    dataloader_num_workers=0,
    group_by_length=False,
)

# Transformers renamed evaluation_strategy -> eval_strategy in newer versions.
training_arg_params = inspect.signature(TrainingArguments.__init__).parameters
if 'eval_strategy' in training_arg_params:
    training_kwargs['eval_strategy'] = 'epoch'
else:
    training_kwargs['evaluation_strategy'] = 'epoch'

training_args = TrainingArguments(**training_kwargs)

head_param_ids = {id(p) for p in model.lm_head.parameters()}
head_params = [p for p in model.parameters() if id(p) in head_param_ids and p.requires_grad]
encoder_params = [p for p in model.parameters() if id(p) not in head_param_ids and p.requires_grad]

optimizer = torch.optim.AdamW(
    [
        {'params': encoder_params, 'lr': ENCODER_LEARNING_RATE},
        {'params': head_params, 'lr': HEAD_LEARNING_RATE},
    ],
    weight_decay=WEIGHT_DECAY,
)

steps_per_epoch = math.ceil(len(train_dataset) / (PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS))
num_training_steps = max(1, int(steps_per_epoch * NUM_EPOCHS))
num_warmup_steps = int(num_training_steps * WARMUP_RATIO)
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

print('encoder trainable params:', sum(p.numel() for p in encoder_params))
print('head trainable params:', sum(p.numel() for p in head_params))
print('encoder lr:', ENCODER_LEARNING_RATE)
print('head lr:', HEAD_LEARNING_RATE)
print('training steps:', num_training_steps, 'warmup steps:', num_warmup_steps)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=data_collator,
    optimizers=(optimizer, lr_scheduler),
)

train_result = trainer.train()
trainer.save_model(str(RUN_DIR / 'final_model'))
feature_extractor.save_pretrained(str(RUN_DIR / 'final_model'))
print(train_result)

## 19. Decode Validation Predictions

In [ ]:
def greedy_decode_logits(logits, blank_penalty=0.0):
    adjusted = logits.copy()
    if blank_penalty:
        adjusted[..., phone2id[blank_token]] -= float(blank_penalty)
    pred_ids = np.argmax(adjusted, axis=-1)
    return [decode_phone_ids(seq.tolist(), collapse_ctc=True) for seq in pred_ids]

def summarize_decode(logits, penalties=(0.0, 0.5, 1.0, 1.5, 2.0)):
    raw_pred_ids = np.argmax(logits, axis=-1)
    frame_counts = Counter(raw_pred_ids.reshape(-1).tolist())
    total_frames = raw_pred_ids.size
    print('Raw greedy blank frame ratio:', frame_counts[phone2id[blank_token]] / total_frames)
    print('Top raw frame tokens:')
    for idx, count in frame_counts.most_common(10):
        print(f'  {id2phone[idx]} ({idx}): {count / total_frames:.4f}')
    print('Decode penalty sweep:')
    for penalty in penalties:
        preds = greedy_decode_logits(logits, blank_penalty=penalty)
        lengths = pd.Series([len(str(p).split()) for p in preds])
        print(
            f'  blank_penalty={penalty:.1f} '
            f'empty={(lengths == 0).sum()}/{len(lengths)} '
            f'mean_len={lengths.mean():.2f} '
            f'median_len={lengths.median():.2f}'
        )

pred_output = trainer.predict(valid_dataset)
summarize_decode(pred_output.predictions)
valid_predictions = greedy_decode_logits(pred_output.predictions, blank_penalty=BLANK_DECODE_PENALTY)

valid_ground_truth = valid_df[['canonical_norm', 'transcript_norm']].rename(
    columns={'canonical_norm': 'canonical', 'transcript_norm': 'transcript'}
).reset_index(drop=True)
valid_results = pd.DataFrame({'predict': valid_predictions})

valid_ground_truth_path = RUN_DIR / 'valid_ground_truth.csv'
valid_results_path = RUN_DIR / 'valid_results.csv'
valid_ground_truth.to_csv(valid_ground_truth_path, index=False)
valid_results.to_csv(valid_results_path, index=False)

debug_decode_df = pd.concat(
    [valid_df[['id', 'transcript_norm']].reset_index(drop=True), valid_results],
    axis=1,
)
debug_decode_df['target_len'] = debug_decode_df['transcript_norm'].str.split().map(len)
debug_decode_df['pred_len'] = debug_decode_df['predict'].fillna('').map(lambda s: len(str(s).split()))
debug_decode_df['is_empty_pred'] = debug_decode_df['pred_len'] == 0
debug_decode_path = RUN_DIR / 'valid_decode_debug.csv'
debug_decode_df.to_csv(debug_decode_path, index=False)

print('Wrote:', valid_ground_truth_path)
print('Wrote:', valid_results_path)
print('Wrote:', debug_decode_path)
print('Selected blank penalty:', BLANK_DECODE_PENALTY)
print('Empty predictions:', int(debug_decode_df['is_empty_pred'].sum()), '/', len(debug_decode_df))
display(debug_decode_df[['target_len', 'pred_len']].describe())
display(pd.concat([valid_df[['id', 'canonical_norm', 'transcript_norm']].reset_index(drop=True), valid_results], axis=1).head(20))

## 20. Evaluate Validation Metrics

These functions mirror `evaluate.py`, so the notebook can score itself on Kaggle even when `evaluate.py` is not uploaded.

In [ ]:
def _read_csv_rows(path):
    with open(path, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

def compute_f1_from_paths(ground_truth_path, results_path):
    gt = _read_csv_rows(ground_truth_path)
    res = _read_csv_rows(results_path)
    cor_cor = cor_nocor = 0
    sub_sub = sub_sub1 = sub_nosub = 0
    ins_ins = ins_ins1 = ins_noins = 0
    del_del = del_del1 = del_nodel = 0
    for gt_row, res_row in zip(gt, res):
        ref_str = gt_row['canonical']
        human_str = gt_row['transcript']
        our_str = res_row['predict']
        ref_seq, human_seq, op_rh = align_pair(ref_str, human_str)
        human_seq2, our_seq2, op_ho = align_pair(human_str, our_str)
        ref_seq3, our_seq3, op_ro = align_pair(ref_str, our_str)
        flag = 0
        for i in range(len(ref_seq)):
            if ref_seq[i] == '<eps>':
                continue
            while flag < len(ref_seq3) and ref_seq3[flag] == '<eps>':
                flag += 1
            if flag < len(ref_seq3) and ref_seq[i] == ref_seq3[flag]:
                if op_rh[i] == 'D' and op_ro[flag] == 'D':
                    del_del += 1
                elif op_rh[i] == 'D' and op_ro[flag] != 'D' and op_ro[flag] != 'C':
                    del_del1 += 1
                elif op_rh[i] == 'D' and op_ro[flag] != 'D' and op_ro[flag] == 'C':
                    del_nodel += 1
                flag += 1
        flag = 0
        for i in range(len(human_seq)):
            if human_seq[i] == '<eps>':
                continue
            while flag < len(human_seq2) and human_seq2[flag] == '<eps>':
                flag += 1
            if flag < len(human_seq2) and human_seq[i] == human_seq2[flag]:
                if op_rh[i] == 'C' and op_ho[flag] == 'C':
                    cor_cor += 1
                elif op_rh[i] == 'C' and op_ho[flag] != 'C':
                    cor_nocor += 1
                if op_rh[i] == 'S' and op_ho[flag] == 'C':
                    sub_sub += 1
                elif op_rh[i] == 'S' and op_ho[flag] != 'C' and ref_seq[i] != our_seq2[flag]:
                    sub_sub1 += 1
                elif op_rh[i] == 'S' and op_ho[flag] != 'C' and ref_seq[i] == our_seq2[flag]:
                    sub_nosub += 1
                if op_rh[i] == 'I' and op_ho[flag] == 'C':
                    ins_ins += 1
                elif op_rh[i] == 'I' and op_ho[flag] != 'C' and op_ho[flag] != 'D':
                    ins_ins1 += 1
                elif op_rh[i] == 'I' and op_ho[flag] != 'C' and op_ho[flag] == 'D':
                    ins_noins += 1
                flag += 1
    TR = sub_sub + sub_sub1 + del_del + del_del1 + ins_ins + ins_ins1
    FR = cor_nocor
    FA = sub_nosub + ins_noins + del_nodel
    precision = TR / (TR + FR) if (TR + FR) > 0 else 0.0
    recall = TR / (TR + FA) if (TR + FA) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return f1, precision, recall

def compute_per_from_paths(ground_truth_path, results_path):
    gt = _read_csv_rows(ground_truth_path)
    res = _read_csv_rows(results_path)
    total_sub = total_del = total_ins = total_ref_len = 0
    for gt_row, res_row in zip(gt, res):
        _, _, op_ho = align_pair(gt_row['transcript'], res_row['predict'])
        total_sub += op_ho.count('S')
        total_del += op_ho.count('D')
        total_ins += op_ho.count('I')
        total_ref_len += op_ho.count('S') + op_ho.count('D') + op_ho.count('C')
    return (total_sub + total_del + total_ins) / total_ref_len if total_ref_len > 0 else 0.0

def compute_der_from_paths(ground_truth_path, results_path):
    gt = _read_csv_rows(ground_truth_path)
    res = _read_csv_rows(results_path)
    sub_sub = sub_sub1 = sub_nosub = 0
    ins_ins = ins_ins1 = ins_noins = 0
    del_del = del_del1 = del_nodel = 0
    for gt_row, res_row in zip(gt, res):
        ref_str = gt_row['canonical']
        human_str = gt_row['transcript']
        our_str = res_row['predict']
        ref_seq, human_seq, op_rh = align_pair(ref_str, human_str)
        human_seq2, our_seq2, op_ho = align_pair(human_str, our_str)
        ref_seq3, our_seq3, op_ro = align_pair(ref_str, our_str)
        flag = 0
        for i in range(len(ref_seq)):
            if ref_seq[i] == '<eps>':
                continue
            while flag < len(ref_seq3) and ref_seq3[flag] == '<eps>':
                flag += 1
            if flag < len(ref_seq3) and ref_seq[i] == ref_seq3[flag]:
                if op_rh[i] == 'D' and op_ro[flag] == 'D':
                    del_del += 1
                elif op_rh[i] == 'D' and op_ro[flag] != 'D' and op_ro[flag] != 'C':
                    del_del1 += 1
                elif op_rh[i] == 'D' and op_ro[flag] != 'D' and op_ro[flag] == 'C':
                    del_nodel += 1
                flag += 1
        flag = 0
        for i in range(len(human_seq)):
            if human_seq[i] == '<eps>':
                continue
            while flag < len(human_seq2) and human_seq2[flag] == '<eps>':
                flag += 1
            if flag < len(human_seq2) and human_seq[i] == human_seq2[flag]:
                if op_rh[i] == 'S' and op_ho[flag] == 'C':
                    sub_sub += 1
                elif op_rh[i] == 'S' and op_ho[flag] != 'C' and ref_seq[i] != our_seq2[flag]:
                    sub_sub1 += 1
                elif op_rh[i] == 'S' and op_ho[flag] != 'C' and ref_seq[i] == our_seq2[flag]:
                    sub_nosub += 1
                if op_rh[i] == 'I' and op_ho[flag] == 'C':
                    ins_ins += 1
                elif op_rh[i] == 'I' and op_ho[flag] != 'C' and op_ho[flag] != 'D':
                    ins_ins1 += 1
                elif op_rh[i] == 'I' and op_ho[flag] != 'C' and op_ho[flag] == 'D':
                    ins_noins += 1
                flag += 1
    TR = sub_sub + sub_sub1 + del_del + del_del1 + ins_ins + ins_ins1
    FA = sub_nosub + ins_noins + del_nodel
    DE = sub_sub1 + del_del1 + ins_ins1
    total_actual_errors = TR + FA
    return DE / total_actual_errors if total_actual_errors > 0 else 0.0

f1, precision, recall = compute_f1_from_paths(valid_ground_truth_path, valid_results_path)
per = compute_per_from_paths(valid_ground_truth_path, valid_results_path)
der = compute_der_from_paths(valid_ground_truth_path, valid_results_path)
score = 0.5 * f1 + 0.4 * (1 - der) + 0.1 * (1 - per)

print(f'F1: {f1:.6f}')
print(f'Precision: {precision:.6f}')
print(f'Recall: {recall:.6f}')
print(f'PER: {per:.6f}')
print(f'DER: {der:.6f}')
print(f'Score: {score:.6f}')

## 21. Run Settings Notes

The default training config above is now intended to produce a real phoneme-CTC baseline instead of a quick sanity run. The old sanity configuration (`USE_SUBSET=True`, `NUM_EPOCHS=1`) is useful only for checking plumbing; it commonly predicts mostly blank because the phoneme CTC head is randomly initialized and CTC has many more blank frames than label frames.

For a quick end-to-end check, set:

```python
FAST_DEV_RUN = True
```

For a stronger run after the baseline works, try:

```python
NUM_EPOCHS = 20
PER_DEVICE_BATCH_SIZE = 4  # if VRAM allows
GRAD_ACCUM_STEPS = 4
ENABLE_AUDIO_AUGMENT = True
BLANK_DECODE_PENALTY = 0.5  # tune on validation: 0.0, 0.5, 1.0, 1.5, 2.0
```

Keep augmentation light and audio-only. Do not crop audio because the CTC label is the full utterance transcript phones.